# Train and Export the RLHF Demo VAE

This notebook runs the same training/export scripts as the local backend commands. It is meant for Google Colab, but it also works in Jupyter if you run it from the repo root.

Expected outputs:

- `backend_training/checkpoints/vae.pt`
- `backend_training/checkpoints/recon_preview.png`
- `backend_training/checkpoints/sample_grid.png`
- `public/models/decoder.onnx`
- `public/models/decoder.onnx.data`
- `public/models/metadata.json`
- `public/models/latent_map.json`


## 1. Runtime Setup

In Colab, choose **Runtime > Change runtime type > T4 GPU** or another GPU runtime before running this notebook.

This notebook expects to run from the repo root. If you opened the notebook from outside the repo tree, copy or clone the repo into Colab first, then point `REPO_DIR` at that checkout.

In [2]:
from pathlib import Path
import os
import sys

# If auto-detection fails in Colab, set REPO_DIR manually to your repo checkout.
CANDIDATE_REPO_DIRS = [
    Path.cwd(),
    Path('/content/vaedemo'),
    Path('/content/drive/MyDrive/vaedemo'),
]

REPO_DIR = None
for candidate in CANDIDATE_REPO_DIRS:
    if (candidate / 'backend_training/train_cvae.py').exists():
        REPO_DIR = candidate
        break

if REPO_DIR is None:
    raise FileNotFoundError(
        'Could not find the repo root. Copy or clone the repo into Colab, then set REPO_DIR manually in this cell.'
    )

os.chdir(REPO_DIR)

print('Working directory:', Path.cwd())
print('Python:', sys.version)


Working directory: /content/vaedemo
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


## 2. Install Python Dependencies

Colab usually already has PyTorch, but this keeps the export dependencies available too.

In [3]:
%pip install -q torch torchvision onnx onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 133.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 19.3 MB/s eta 0:00:00


## 3. Verify Repo Files

These checks make sure the notebook is using the same implementation as the local commands.

In [4]:
required_files = [
    Path('backend_training/data/colored_mnist.py'),
    Path('backend_training/data/mnist_rgb.py'),
    Path('backend_training/models/vae.py'),
    Path('backend_training/train_cvae.py'),
    Path('backend_training/export_decoder.py'),
    Path('backend_training/sample_grid.py'),
]

missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError('Missing repo files: ' + ', '.join(missing))

print('All backend training files found.')


All backend training files found.


## 4. Train the VAE

This uses the exact same script as local training. The defaults are `latent_dim=2` and `image_size=64`.

For a quick smoke test, set `EPOCHS = 1`. For a usable demo, start with `EPOCHS = 20`.

In [6]:
import torch

EPOCHS = 20
BATCH_SIZE = 256
NUM_WORKERS = 2
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Device:', DEVICE)
!python backend_training/train_cvae.py --epochs {EPOCHS} --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS} --device {DEVICE}


Device: cuda
epoch 001 loss=0.15073 recon=0.14872 kl=2.01063
epoch 002 loss=0.08338 recon=0.08098 kl=2.40797
epoch 003 loss=0.07766 recon=0.07505 kl=2.61742
epoch 004 loss=0.07552 recon=0.07282 kl=2.70211
epoch 005 loss=0.07421 recon=0.07145 kl=2.75984
epoch 006 loss=0.07297 recon=0.07016 kl=2.81587
epoch 007 loss=0.07211 recon=0.06925 kl=2.85947
epoch 008 loss=0.07158 recon=0.06868 kl=2.89943
epoch 009 loss=0.07108 recon=0.06813 kl=2.94859
epoch 010 loss=0.07048 recon=0.06749 kl=2.99375
epoch 011 loss=0.06989 recon=0.06686 kl=3.02891
epoch 012 loss=0.06956 recon=0.06650 kl=3.05975
epoch 013 loss=0.06897 recon=0.06589 kl=3.08105
epoch 014 loss=0.06868 recon=0.06558 kl=3.10127
epoch 015 loss=0.06842 recon=0.06528 kl=3.13463
epoch 016 loss=0.06813 recon=0.06499 kl=3.14527
epoch 017 loss=0.06781 recon=0.06464 kl=3.17127
epoch 018 loss=0.06747 recon=0.06429 kl=3.18184
epoch 019 loss=0.06739 recon=0.06419 kl=3.19785
epoch 020 loss=0.06724 recon=0.06403 kl=3.21520
saved checkpoint: backend_t

## 5. Export the Decoder and Latent Map for the Website

This writes the same frontend files used by the Vite app.

In [ ]:
!python backend_training/export_decoder.py

ModuleNotFoundError: No module named 'data'

## 6. Sample a Latent Grid for Visual Inspection

This image is just for checking decoder quality before you use the website.

In [ ]:
!python backend_training/sample_grid.py --out backend_training/checkpoints/sample_grid.png --device {DEVICE}


Traceback (most recent call last):
  File "/content/vaedemo/backend_training/sample_grid.py", line 49, in <module>
    main()
  File "/content/vaedemo/backend_training/sample_grid.py", line 23, in main
    checkpoint = torch.load(args.checkpoint, map_location=args.device)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1500, in load
    with _open_file_like(f, "rb") as opened_file:
         ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 768, in _open_file_like
    return _open_file(name_or_buffer, mode)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 749, in __init__
    super().__init__(open(name, mode))  # noqa: SIM115
                     ^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'backend_training/checkpoints/mnist_vae.pt'


In [ ]:
from IPython.display import Image, display

preview = Path('backend_training/checkpoints/sample_grid.png')
if preview.exists():
    display(Image(filename=str(preview)))
else:
    print('Preview image not found:', preview)


Preview image not found: backend_training/checkpoints/sample_grid.png


## 7. Confirm Output Files

Copy `public/models/decoder.onnx`, `public/models/decoder.onnx.data`, `public/models/metadata.json`, and `public/models/latent_map.json` back into the local repo if you trained in Colab.

Those three files are the handoff into the local RLHF demo. The browser keeps the decoder frozen and learns a reward model and latent policy from local preference clicks.

In [ ]:
outputs = [
    Path('backend_training/checkpoints/vae.pt'),
    Path('backend_training/checkpoints/recon_preview.png'),
    Path('backend_training/checkpoints/sample_grid.png'),
    Path('public/models/decoder.onnx'),
    Path('public/models/decoder.onnx.data'),
    Path('public/models/metadata.json'),
    Path('public/models/latent_map.json'),
]

for path in outputs:
    status = 'OK' if path.exists() else 'MISSING'
    size = path.stat().st_size if path.exists() else 0
    print(f'{status:7} {size:>12} bytes  {path}')


OK           5662460 bytes  backend_training/checkpoints/vae.pt
OK            104640 bytes  backend_training/checkpoints/recon_preview.png
MISSING            0 bytes  backend_training/checkpoints/sample_grid.png
OK              3657 bytes  public/models/decoder.onnx
OK           3014656 bytes  public/models/decoder.onnx.data
OK               169 bytes  public/models/metadata.json
OK            148241 bytes  public/models/latent_map.json


## Optional: Zip the Website Model Files

This makes it easier to download the exact files needed by the frontend. In Colab, the next cell also triggers a browser download.

In [ ]:
zip_path = Path('public/models/frontend_decoder_assets.zip')
!zip -j {zip_path} public/models/decoder.onnx public/models/decoder.onnx.data public/models/metadata.json public/models/latent_map.json
print(f'Created {zip_path}')

try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print('Not running in Colab. Download the zip from the local filesystem if needed.')


updating: decoder.onnx (deflated 56%)
updating: decoder.onnx.data (deflated 8%)
updating: metadata.json (deflated 38%)
  adding: latent_map.json (deflated 85%)
Created public/models/frontend_decoder_assets.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download('public/models/frontend_decoder_assets.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>